# 11 · Hyperparameter optimization

## Purpose

Run a reproducible, bounded Optuna search for a selected installed gradient-boosting model.

## Method

Use Quick or Full mode, training folds only, a seeded TPE sampler, early stopping where supported by the installed estimator, and persist every trial and its duration.

## Interpretation

Use the displayed tables, figures, and saved metrics produced by this run. Do not fill this section with assumed outcomes. For every result, distinguish observed evidence from interpretation.

## Limitations

A search is meaningful only when a supported boosting family is installed. The untouched internal final holdout is not accessed by this notebook.

## References

Optuna documentation: https://optuna.readthedocs.io/en/stable/


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'returnwise').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from returnwise.research_state import load_state, save_state
state = load_state()
print('Project:', ROOT)
print('Notebook:', '11 · Hyperparameter optimization')


Project: C:\Users\Lenovo\Documents\Who is back
Notebook: 11 · Hyperparameter optimization


In [2]:
from returnwise.models import model_registry, tune_with_optuna
from returnwise.research_state import load_state, save_state
import matplotlib.pyplot as plt
import pandas as pd

state = load_state()
cv_parts = [x for x in [state.get('baseline_results'), state.get('advanced_results')] if x is not None]
cv_results = pd.concat(cv_parts, ignore_index=True) if cv_parts else pd.DataFrame()
if cv_results.empty:
    raise RuntimeError('Run notebooks 09 and 10 first.')
registry, skipped = model_registry(state['X_train'], mode=state.get('experiment_mode', 'quick'))
boosting = [name for name in ['CatBoost', 'LightGBM', 'XGBoost'] if name in registry]
completed = cv_results[cv_results['status'] == 'completed'].sort_values('roc_auc', ascending=False)
best_cv_model = completed.iloc[0]['model'] if not completed.empty else None
if not boosting:
    print('No optional gradient-boosting library is available; optimization was not run.')
    print('Availability:', skipped)
    tuning_result = None
else:
    if best_cv_model not in boosting:
        print('The best CV candidate is not an installed modern boosting model; no Optuna run was performed.')
        tuning_result = None
    else:
        tuning_candidate = best_cv_model
        try:
            tuning_result = tune_with_optuna(state['X_train'], state['y_train'], tuning_candidate, mode=state.get('experiment_mode', 'quick'), seed=42)
            print('Selected search candidate:', tuning_candidate)
            print('Best training-CV ROC-AUC:', tuning_result['best_score_roc_auc'])
            print('Duration (seconds):', tuning_result['duration_seconds'])
            trials = pd.DataFrame(tuning_result['trials'])
            display(trials)
            ax = trials.sort_values('number').plot(x='number', y='value', marker='o', figsize=(7, 3), legend=False, title='Optuna trial ROC-AUC')
            ax.set_ylabel('Cross-validated ROC-AUC')
            plt.tight_layout()
            plt.show()
        except RuntimeError as error:
            print('Optuna search skipped:', error)
            tuning_result = None
state['tuning_result'] = tuning_result
from returnwise.experiments import log_experiment
if tuning_result:
    log_experiment('Optuna hyperparameter search', None,
                   {'best_score_roc_auc': tuning_result['best_score_roc_auc'], 'trials': tuning_result['trials']},
                   {'model': tuning_result['model'], 'best_parameters': tuning_result['best_parameters'],
                    'duration_seconds': tuning_result['duration_seconds']})
save_state(state)


No optional gradient-boosting library is available; optimization was not run.
Availability: {'CatBoost': 'Optional dependency `catboost` is not installed.', 'LightGBM': 'Optional dependency `lightgbm` is not installed.', 'XGBoost': 'Optional dependency `xgboost` is not installed.', 'TabPFN': 'Optional dependency `tabpfn` is not installed.'}
